# trRosettaRNA2 — Google Colab Verification Notebook

**Project:** RNA 3D Structure Prediction for Template-Sparse RNA Sequences  
**Model:** trRosettaRNA2  
**Official repo:** `YangLab-SDU/trRosettaRNA2`

This notebook is Colab-first. The initial verification uses the bundled/precomputed A3M, so the optional ~32 GB RNAcentral database is not required.

Verified final output: `example/output/model_1_relaxed200.pdb`.

## 1. GPU and host check

In [ ]:
import os, sys, time, threading, subprocess, platform
from pathlib import Path

print("Host Python:", sys.version)
print("Platform:", platform.platform())
subprocess.run(["nvidia-smi"], check=False)

## 2. Clone the official repository

In [ ]:
%cd /content
if not Path("/content/trRosettaRNA2").exists():
    !git clone https://github.com/YangLab-SDU/trRosettaRNA2.git
else:
    print("Repository already exists; reusing it.")

%cd /content/trRosettaRNA2
commit = subprocess.check_output(["git","rev-parse","HEAD"], text=True).strip()
print("trRosettaRNA2 commit:", commit)

## 3. Inspect the official environment

In [ ]:
%cd /content/trRosettaRNA2
!cat environment.yml

## 4. Install Micromamba

In [ ]:
%cd /content
if not Path("/content/bin/micromamba").exists():
    !curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj
!/content/bin/micromamba --version

## 5. Create the official environment

The repository environment is named `trRNA2` and includes Python 3.10, PyTorch 2.5.1 CUDA 11.8 build, PyRosetta, NumPy 2.1.2 and the remaining model dependencies.

In [ ]:
%cd /content/trRosettaRNA2

envs = subprocess.run(
    ["/content/bin/micromamba","-r","/content/micromamba_root","env","list"],
    capture_output=True, text=True
)

if "trRNA2" not in envs.stdout:
    subprocess.run([
        "/content/bin/micromamba","-r","/content/micromamba_root",
        "create","-y","-n","trRNA2","-f","/content/trRosettaRNA2/environment.yml"
    ], check=True)
else:
    print("trRNA2 environment already exists.")

## 6. Verify environment

In [ ]:
verify_code = """
import sys, numpy, torch
print("Python:", sys.version)
print("NumPy:", numpy.__version__)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
"""

r = subprocess.run([
    "/content/bin/micromamba","-r","/content/micromamba_root",
    "run","-n","trRNA2","python","-c",verify_code
], capture_output=True, text=True)

print("RETURN CODE:", r.returncode)
print("STDOUT:\n", r.stdout)
print("STDERR:\n", r.stderr)

if r.returncode != 0:
    raise RuntimeError("Environment verification failed.")

## 7. Download official network weights

Primary source is the YangLab server. If it fails, the Hugging Face mirror is used.

In [ ]:
%cd /content/trRosettaRNA2

archive = Path("/content/params_trRNA2.tar.bz2")

if not archive.exists():
    primary = "http://yanglab.qd.sdu.edu.cn/trRosettaRNA/download/params_trRNA2.tar.bz2"
    mirror = "https://huggingface.co/datasets/quailwwk/trRNA2/resolve/main/params_trRNA2.tar.bz2"

    p = subprocess.run(["wget","-O",str(archive),primary])
    if p.returncode != 0 or not archive.exists() or archive.stat().st_size == 0:
        if archive.exists():
            archive.unlink()
        subprocess.run(["wget","-O",str(archive),mirror], check=True)

print("Archive size GB:", round(archive.stat().st_size/(1024**3),3))

## 8. Extract weights

In [ ]:
%cd /content/trRosettaRNA2

marker = Path("/content/trRosettaRNA2/.weights_extracted")
if not marker.exists():
    subprocess.run(
        ["tar","-xjf","/content/params_trRNA2.tar.bz2"],
        cwd="/content/trRosettaRNA2",
        check=True
    )
    marker.write_text("ok\n")
else:
    print("Weights already extracted.")

## 9. Inspect bundled example input

The standalone pipeline uses A3M MSA input. This verification run uses `example/seq.a3m` and does not download the optional ~32 GB RNAcentral database.

In [ ]:
%cd /content/trRosettaRNA2

print("Example files:")
!find example -maxdepth 3 -type f | sort

msa_path = Path("/content/trRosettaRNA2/example/seq.a3m")
if not msa_path.exists():
    raise FileNotFoundError("example/seq.a3m not found")

print("\nMSA found:", msa_path)

## 10. Check prediction command

If no custom secondary structure is supplied, trRosettaRNA2 uses its internal `trRNA2-SS` predictor.

In [ ]:
%cd /content/trRosettaRNA2

h = subprocess.run([
    "/content/bin/micromamba","-r","/content/micromamba_root",
    "run","-n","trRNA2","python","-m","trRNA2.predict","-h"
], capture_output=True, text=True)

print(h.stdout)
print(h.stderr)

if h.returncode != 0:
    raise RuntimeError("Could not load trRNA2.predict")

## 11. Run official example + log runtime and observed GPU memory

Official default:

```bash
python -m trRNA2.predict -i example/seq.a3m -o example/output
```

In [ ]:
%cd /content/trRosettaRNA2

gpu_samples = []
stop_monitor = False

def monitor_gpu():
    global stop_monitor
    while not stop_monitor:
        try:
            out = subprocess.check_output(
                ["nvidia-smi","--query-gpu=memory.used","--format=csv,noheader,nounits"],
                text=True
            ).strip().splitlines()
            if out:
                gpu_samples.append(int(out[0]))
        except Exception:
            pass
        time.sleep(0.5)

t = threading.Thread(target=monitor_gpu, daemon=True)
t.start()
start = time.perf_counter()

proc = subprocess.run([
    "/content/bin/micromamba","-r","/content/micromamba_root",
    "run","-n","trRNA2",
    "python","-m","trRNA2.predict",
    "-i","example/seq.a3m",
    "-o","example/output"
], cwd="/content/trRosettaRNA2", capture_output=True, text=True)

elapsed = time.perf_counter() - start
stop_monitor = True
t.join(timeout=2)

peak_gpu_mb = max(gpu_samples) if gpu_samples else None

print("===== STDOUT =====")
print(proc.stdout)
print("===== STDERR =====")
print(proc.stderr)
print("=== RUN SUMMARY ===")
print("Exit code:", proc.returncode)
print(f"Runtime: {elapsed:.2f} s")
print("Peak observed GPU memory:", f"{peak_gpu_mb} MiB" if peak_gpu_mb is not None else "Unavailable")

if proc.returncode != 0:
    raise RuntimeError("Inference failed; check STDERR.")

## 12. Verify generated PDB

In [ ]:
expected_pdb = Path("/content/trRosettaRNA2/example/output/model_1_relaxed200.pdb")

print("Expected PDB:", expected_pdb)
print("Exists:", expected_pdb.exists())

for p in sorted(Path("/content/trRosettaRNA2/example/output").rglob("*")):
    if p.is_file():
        print(p)

if not expected_pdb.exists():
    raise FileNotFoundError("Expected model_1_relaxed200.pdb was not generated.")

## 13. Save reproducibility metadata

In [ ]:
import csv

version_code = """
import sys, numpy, torch
print(sys.version.split()[0])
print(numpy.__version__)
print(torch.__version__)
"""

versions = subprocess.check_output([
    "/content/bin/micromamba","-r","/content/micromamba_root",
    "run","-n","trRNA2","python","-c",version_code
], text=True).strip().splitlines()

gpu_name = subprocess.check_output(
    ["nvidia-smi","--query-gpu=name","--format=csv,noheader"],
    text=True
).strip().splitlines()[0]

record = {
    "model":"trRosettaRNA2",
    "model_commit":commit,
    "input":"example/seq.a3m",
    "output":str(expected_pdb),
    "python":versions[0] if len(versions)>0 else "",
    "numpy":versions[1] if len(versions)>1 else "",
    "pytorch":versions[2] if len(versions)>2 else "",
    "runtime_seconds":round(elapsed,3),
    "gpu":gpu_name,
    "peak_gpu_memory_mib_observed":peak_gpu_mb,
    "exit_code":proc.returncode
}

log_path = Path("/content/trrosettarna2_run_log.csv")
with log_path.open("w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=record.keys())
    w.writeheader()
    w.writerow(record)

print(record)
print("Saved:", log_path)

## 14. Optional — save to Google Drive

In [ ]:
# OPTIONAL CELL
# from google.colab import drive
# import shutil
#
# drive.mount("/content/drive")
# destination = Path("/content/drive/MyDrive/RNA3D_Benchmark/trRosettaRNA2")
# destination.mkdir(parents=True, exist_ok=True)
# shutil.copy2(expected_pdb, destination / expected_pdb.name)
# shutil.copy2(log_path, destination / log_path.name)
# print("Saved to:", destination)

## 15. Benchmark policy

Primary benchmark:
- use a target-specific A3M generated with one standardized MSA policy
- use internal `trRNA2-SS`
- do not use native 3D structure during inference
- native structure is evaluation-only
- custom SS should be a separate ablation/oracle condition
- record commit, runtime, GPU and peak VRAM

For team reproducibility, share/precompute A3M files rather than forcing every member to download the optional ~32 GB RNAcentral database.